In [145]:
import warnings
warnings.filterwarnings('ignore')

In [146]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process
import ftfy

from zipfile import ZipFile
from urllib.request import urlopen
from io import BytesIO

In [147]:
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQZC7cVru6ltLR2e8XN5jdJPfxfj42BAxUApe3Zq3_ENQjLtYntmAxD0pIHqEUJ4ZFLXlybKJdkLf2r/pub?output=csv'
candinfo = pd.read_csv(url)
candinfo.to_csv('../../2026_data/2026_midterms_candidateinfo.csv')
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any
0,AK-AL,NaN,Nick Begich,False,True
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False
2,AL-02,Shomari Figures,Hampton Harris,True,False
3,AL-03,Lee McInnis,Mike Rogers,False,True
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True


In [148]:
for party in ['dem', 'rep']:
    candinfo[f'{party}_cand'] = candinfo[f'{party}_cand'].fillna('TBD').astype(str)
    candinfo[f'{party}_inc_any'] = candinfo[f'{party}_inc_any'].astype(bool)

In [149]:
dem_only_candinfo = candinfo[candinfo['rep_cand'] == 'Not Contested']
rep_only_candinfo = candinfo[candinfo['dem_cand'] == 'Not Contested']
candinfo = candinfo[(candinfo['dem_cand'] != 'Not Contested') &
    (candinfo['rep_cand'] != 'Not Contested')]

In [150]:
candinfo['state_po'] = candinfo['cd'].map(lambda x: x[:2]).astype(str)
candinfo['district_number'] = candinfo['cd'].map(lambda x: 0 if x[3:] == 'AL' else int(x[3:]))
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number
0,AK-AL,TBD,Nick Begich,False,True,AK,0
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4


In [151]:
candinfo.shape

(423, 7)

In [152]:
# Code snippet courtesy of hantoine: https://gist.github.com/hantoine/c4fc70b32c2d163f604a8dc2a050d5f6
def download_and_unzip(url, extract_to='.'):
    http_candinfoponse = urlopen(url)
    zipfile = ZipFile(BytesIO(http_candinfoponse.read()))
    zipfile.extractall(path=extract_to)

In [153]:
fec_webl_url = 'https://www.fec.gov/files/bulk-downloads/2026/webl26.zip'
fec_cn_url = 'https://www.fec.gov/files/bulk-downloads/2026/cn26.zip'
download_and_unzip(fec_webl_url, extract_to='../../2026_data/fec')
download_and_unzip(fec_cn_url, extract_to='../../2026_data/fec')

In [154]:
fec_webl_colnames = ["CAND_ID", "CAND_NAME", "CAND_ICI", "PTY_CD", "CAND_PTY_AFFILIATION", "TTL_RECEIPTS", "TRANS_FROM_AUTH", "TTL_DISB", "TRANS_TO_AUTH", "COH_BOP", "COH_COP", "CAND_CONTRIB", "CAND_LOANS", "OTHER_LOANS", "CAND_LOAN_REPAY", "OTHER_LOAN_REPAY", "DEBTS_OWED_BY", "TTL_INDIV_CONTRIB", "CAND_OFFICE_ST", "CAND_OFFICE_DISTRICT", "SPEC_ELECTION", "PRIM_ELECTION", "RUN_ELECTION", "GEN_ELECTION", "GEN_ELECTION_PRECENT", "OTHER_POL_CMTE_CONTRIB", "POL_PTY_CONTRIB", "CVG_END_DT", "INDIV_REFUNDS", "CMTE_REFUNDS"]

In [155]:
fec_cn_colnames = ["CAND_ID", "CAND_NAME", "CAND_PTY_AFFILIATION", "CAND_ELECTION_YR", "CAND_OFFICE_ST", "CAND_OFFICE", "CAND_OFFICE_DISTRICT", "CAND_ICI", "CAND_STATUS", "CAND_PCC", "CAND_ST1", "CAND_ST2", "CAND_CITY", "CAND_ST", "CAND_ZIP"]

In [156]:
webl = pd.read_table('../../2026_data/fec/webl26.txt', sep="|", names=fec_webl_colnames)
cn = pd.read_table('../../2026_data/fec/cn.txt', sep="|", names=fec_cn_colnames)
fec = pd.merge(left=webl, right=cn, on='CAND_ID', how='left')
fec = fec[[col for col in fec.columns.values if ('_y' not in col)]]
fec.columns = fec.columns.str.strip('_x')
fec.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CMTE_REFUNDS,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,H2AK01158,"PELTOLA, MARY",C,1,DEM,152304.86,0.00,232791.29,0.0,83969.49,...,0.00,2026.0,H,C,C00812388,810 N STREET,SUITE 301,ANCHORAGE,AK,99501.0
1,H6AK01084,"SCHULTZ, MATTHEW DAMIAN",C,1,DEM,579656.42,0.00,231366.34,1075.0,0.00,...,0.00,2026.0,H,C,C00923714,PO BOX 240641,NaN,ANCHORAGE,AK,99524.0
2,H2AK01083,"BEGICH, NICHOLAS III",I,2,REP,4307322.69,1302998.38,1581037.44,0.0,104330.06,...,3382.86,2026.0,H,C,C00792341,PO BOX 671710,NaN,CHUGIAK,AK,99567.0
3,H6AK01092,"HILL, BILL",C,3,IND,783044.48,0.00,187786.59,0.0,0.00,...,0.00,2026.0,H,C,C00935437,PO BOX 220703,NaN,ANCHORAGE,AK,99522.0
4,H6AL01094,"JONES, CLYDE W MR. JR",O,1,DEM,37487.19,0.00,18859.98,0.0,0.00,...,0.00,2026.0,H,C,C00920918,11637 WENTWOOD CT,NaN,DAPHNE,AL,36526.0


In [157]:
fec.columns.values

array(['CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD',
       'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS', 'TRANS_FROM_AUTH',
       'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP', 'COH_COP', 'CAND_CONTRIB',
       'CAND_LOANS', 'OTHER_LOANS', 'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY',
       'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST',
       'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
       'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT',
       'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB', 'CVG_END_DT',
       'INDIV_REFUNDS', 'CMTE_REFUNDS', 'CAND_ELECTION_YR', 'CAND_OFFICE',
       'CAND_STATUS', 'CAND_PCC', 'CAND_ST1', 'CAND_ST2', 'CAND_CITY',
       'CAND_ST', 'CAND_ZIP'], dtype=object)

In [158]:
fec = fec[fec['CAND_OFFICE'] == 'H']
fec['CAND_OFFICE_ST'] = fec['CAND_OFFICE_ST'].astype(str)
fec['CAND_OFFICE_DISTRICT'] = fec['CAND_OFFICE_DISTRICT'].astype(float)
fec['CAND_NAME'] = fec['CAND_NAME'].astype(str)

In [159]:
fec.shape

(2334, 39)

In [160]:
def get_fuzzymatch_cand(state_po, district, party, candidate):
    error_return = 'no_match'
    if candidate == 'TBD':
        return error_return
    
    df = fec[(fec['CAND_OFFICE_ST'] == state_po) &
        (fec['CAND_OFFICE_DISTRICT'] == district) &
        (fec['CAND_PTY_AFFILIATION'].isin([party, 'IND', 'NON', 'OTH']))]

    # df['cand_name_lst'] = df['CAND_NAME'].str.split(',')
    # df['first_name_cand'] = df['cand_name_lst'].map(lambda x: x[1])
    # df['last_name_cand'] = df['cand_name_lst'].map(lambda x: x[0])
    # df['cand_name_ordered'] = df['first_name_cand'] + ' ' + df['last_name_cand']

    if df.shape[0] == 0:
        return error_return

    if '/' in candidate:
        cands = candidate.split(separator='/')
        matches, contribs = [], []
        for c in cands:
            fuzzymatch = process.extractOne(c, df['CAND_NAME'].values, scorer=fuzz.WRatio, score_cutoff=55)
            indiv_contrib = df[df['CAND_NAME'] == fuzzymatch]['TTL_INDIV_CONTRIB'].values[0]
            if fuzzymatch is not None:
                matches.append(fuzzymatch[0])
                contribs.append(indiv_contrib)
            else:
                matches.append(error_return)
                contribs.append(0)
        return matches

    else:
        fuzzymatch = process.extractOne(candidate, df['CAND_NAME'].values, scorer=fuzz.token_sort_ratio, score_cutoff=10)
        if fuzzymatch is not None:
            indiv_contrib = df[df['CAND_NAME'] == fuzzymatch[0]]['TTL_INDIV_CONTRIB'].values[0]
        else:
            indiv_contrib = 0
        return error_return if fuzzymatch is None else fuzzymatch[0]

In [161]:
candinfo['fec_dem_fuzzymatch'] = candinfo.apply(lambda x: get_fuzzymatch_cand(x['state_po'], x['district_number'], 'DEM', x['dem_cand']), axis=1)
candinfo['fec_rep_fuzzymatch'] = candinfo.apply(lambda x: get_fuzzymatch_cand(x['state_po'], x['district_number'], 'REP', x['rep_cand']), axis=1)
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch
0,AK-AL,TBD,Nick Begich,False,True,AK,0,no_match,"BEGICH, NICHOLAS III"
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR"
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON"
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL"
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","BARNES, THOMAS GARY"


In [162]:
dem_corrections = {
    ("CA-24", "Salud Carbajal"):    "CARBAJAL, SALUD O.",
    ("GA-01", "Amanda Hollowell"):  "HOLLOWELL, AMANDA",
    ("GA-07", "Tony Kozycki"):      "KOZYCKI, ANTHONY LAWRENCE",
    ("IL-08", "Melissa Bean"):      "BEAN, MELISSA LUBURICH",
    ("IN-09", "Brad Meyer"):        "MEYER, BRADLEY ALLEN MR.",
    ("MT-02", "Brian Miller"):      "MILLER, BRIAN JAMES",
    ("NC-12", "Alma Adams"):        "ADAMS, ALMA SHEALEY",
    ("NJ-07", "Rebecca Bennett"):   "BENNETT, REBECCA",
    ("NJ-11", "Analilia Mejia"):    "MEJIA, ANALILIA",
    ("NJ-12", "Adam Hamawy"):       "HAMAWY, ADAM",
    ("NY-12", "Micah Lasher"):      "LASHER, MICAH CHARLES",
    ("OH-15", "Don Leonard"):       "LEONARD, DON RALPH",
    ("PA-01", "Bob Harvie"):        "HARVIE, ROBERT J",
    ("TN-09", "Justin Pearson"):    "PEARSON, JUSTIN J.",
    ("TX-10", "Caitlin Rourk"):     "ROURK, CAITLIN MCCLAY",
    ("VA-08", "Don Beyer"):         "BEYER, DONALD STERNOFF JR.",
    ("TN-07", "Joshua Sales"):      "no_match",
    ("KY-02", "Megan Wingfield"):   "no_match",
    ("OH-05", "Brian Shaver"):      "no_match",
    ("OH-06", "Elizabeth Kirtley"): "no_match",
    ("OK-03", "Suzie Byrd"):        "no_match",
    ("TX-25", "Dione Sims"):        "no_match",
    ("MA-09", "Bill Keating"):      "KEATING, WILLIAM R",
}

rep_corrections = {
    ("AL-04", "Robert Aderholt"):      "ADERHOLT, ROBERT B. REP.",
    ("FL-01", "Jimmy Patronis"):       "PATRONIS, JIMMY JR.",
    ("GA-12", "Rick Allen"):           "ALLEN, RICHARD W",
    ("KY-06", "Ralph Alvarado"):       "ALVARADO, RALPH A",
    ("PA-05", "Nicholas Manganaro"):   "MANGANARO, NICHOLAS WALN MORRIS",
    ("TX-09", "Alex Mealer"):          "MEALER, ALEXANDRA",
    ("TX-21", "Mark Teixeira"):        "TEIXEIRA, MARK CHARLES",
    ("TX-22", "Trever Nehls"):         "NEHLS, TREVER",
    ("CA-06", "Kevin Kiley"):          "no_match",
    ("IL-04", "Lupe Castillo"):        "no_match",
    ("NY-15", "Stylo Sapaskis"):       "no_match",
    ("CA-02", "Robin Littau"):         "no_match",
    ("IL-09", "John Elleson"):         "no_match",
    ("KS-03", "Chase LaPorte"):        "no_match",
    ("MD-05", "Chris Chaffee"):        "no_match",
    ("NY-01", "Nick LaLota"):          "no_match",
    ("NY-09", "Joel Anabilah-Azumah"): "no_match",
    ("PA-04", "Aurora Stuski"):        "no_match",
    ("TX-18", "Ronald Whitfield"):     "no_match",
    ("MI-04", "Bill Huizenga"):        "HUIZENGA, WILLIAM P",
}

In [163]:
for (cd, cand) in dem_corrections.keys():
    mask = (
        (candinfo['cd'] == cd) &
        (candinfo['dem_cand'] == cand)
    )

    candinfo.loc[mask, 'fec_dem_fuzzymatch'] = corrections[(cd, cand)]

for (cd, cand) in rep_corrections.keys():
    mask = (
        (candinfo['cd'] == cd) &
        (candinfo['rep_cand'] == cand)
    )

    candinfo.loc[mask, 'fec_rep_fuzzymatch'] = corrections[(cd, cand)]

In [164]:
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch
0,AK-AL,TBD,Nick Begich,False,True,AK,0,no_match,"BEGICH, NICHOLAS III"
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR"
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON"
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL"
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP."


In [172]:
def get_indiv_contribs(state_po, district_number, party, candidate):
    if candidate == 'TBD':
        return 0
    
    df = fec[
        (fec['CAND_OFFICE_ST'] == state_po) &
        (fec['CAND_OFFICE_DISTRICT'] == district_number) &
        (fec['CAND_PTY_AFFILIATION'].isin([party, 'IND', 'OTH', 'NON'])) &
        (fec['CAND_NAME'] == candidate)
    ]

    if df.shape[0] == 0:
        return 0

    return df['TTL_INDIV_CONTRIB'].values[0]

candinfo['dem_funds'] = candinfo.apply(lambda x: get_indiv_contribs(x['state_po'], x['district_number'], 'DEM', x['fec_dem_fuzzymatch']), axis=1)
candinfo['rep_funds'] = candinfo.apply(lambda x: get_indiv_contribs(x['state_po'], x['district_number'], 'REP', x['fec_rep_fuzzymatch']), axis=1)
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds
0,AK-AL,TBD,Nick Begich,False,True,AK,0,no_match,"BEGICH, NICHOLAS III",0.00,2101711.89
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",37487.19,483437.38
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",237170.80,8958.91
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",39858.33,1168569.57
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",13708.00,385094.98


In [174]:
candinfo['2p_funds'] = candinfo['dem_funds'] + candinfo['rep_funds']
candinfo['dem_funds_2p_pct'] = candinfo['dem_funds'] / candinfo['2p_funds'] * 100
candinfo['rep_funds_2p_pct'] = candinfo['rep_funds'] / candinfo['2p_funds'] * 100
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct
0,AK-AL,TBD,Nick Begich,False,True,AK,0,no_match,"BEGICH, NICHOLAS III",0.00,2101711.89,2101711.89,0.000000,100.000000
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1,"JONES, CLYDE W MR. JR","CARL, JERRY LEE, JR",37487.19,483437.38,520924.57,7.196280,92.803720
2,AL-02,Shomari Figures,Hampton Harris,True,False,AL,2,"FIGURES, SHOMARI C.","HARRIS, HAMPTON",237170.80,8958.91,246129.71,96.360086,3.639914
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3,"MCINNIS, VICTOR LEE","ROGERS, MICHAEL",39858.33,1168569.57,1208427.90,3.298362,96.701638
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4,"PUSCZEK, AMANDA NOELLE","ADERHOLT, ROBERT B. REP.",13708.00,385094.98,398802.98,3.437286,96.562714


In [181]:
## Wrangle PVI + demographic data for redistricted states
redist_dfs_2020 = []
redist_dfs_2024 = []

redist_states = ['CA', 'UT', 'TN', 'TX', 'LA', 'MO', 'FL', 'AL', 'NC', 'OH']
for st in redist_states:
    print(st)
    dat_2020 = pd.read_csv(f'../../2026_data/dra/{st}_2026_distdata_2020_elec.csv').iloc[1:]
    dat_2024 = pd.read_csv(f'../../2026_data/dra/{st}_2026_distdata_2024_elec.csv').iloc[1:]
    redist_dfs_2020.append(dat_2020[['Label', 'V_24_CVAP_Total', 'V_24_CVAP_White', 'V_24_CVAP_Hispanic', 'V_24_CVAP_BlackAlone', 'V_24_CVAP_AsianAlone',
                               'V_24_CVAP_NativeAlone', 'V_24_CVAP_PacificAlone', 'E_20_PRES_Dem', 'E_20_PRES_Rep', 'X_22_2022_Education_Bach',
                               'X_22_2022_Education_Master', 'X_22_2022_Education_Prof', 'X_22_2022_Education_Doc']])
    redist_dfs_2024.append(dat_2024[['Label', 'V_24_CVAP_Total', 'V_24_CVAP_White', 'V_24_CVAP_Hispanic', 'V_24_CVAP_BlackAlone', 'V_24_CVAP_AsianAlone',
                               'V_24_CVAP_NativeAlone', 'V_24_CVAP_PacificAlone', 'E_24_PRES_Dem', 'E_24_PRES_Rep', 'X_22_2022_Education_Bach',
                               'X_22_2022_Education_Master', 'X_22_2022_Education_Prof', 'X_22_2022_Education_Doc']])

redist_df_20 = pd.concat(redist_dfs_2020, axis=0)
redist_df_24 = pd.concat(redist_dfs_2024, axis=0)

CA
UT
TN
TX
LA
MO
FL
AL
NC
OH
